# Phase 2: Transformer Scaling Study
## SVG Scaling Laws — CS-GY 6923

**Steps:**
1. Mount Drive & clone/pull repo
2. Install dependencies
3. Configure paths
4. Print model parameter counts
5. Run LR sweep (Tiny, 2000 steps × 7 LRs)
6. Update best LR in config
7. Train each of the 5 models individually
8. Fit scaling law & generate all plots

---
## Cell 0: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/svg-scaling-laws'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted. Project dir: {DRIVE_DIR}')

---
## Cell 1: Clone / Pull Repository

In [ ]:
REPO_URL = 'https://github.com/YOUR_USERNAME/svg-scaling-laws.git'  # <-- EDIT THIS
REPO_DIR = '/content/svg-scaling-laws'

import os
if os.path.exists(REPO_DIR):
    print('Repo already exists, pulling latest ...')
    !cd {REPO_DIR} && git pull
else:
    print('Cloning repo ...')
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
print(f'Working directory: {os.getcwd()}')

---
## Cell 2: Install Dependencies

In [ ]:
!pip install -q -r requirements.txt
print('Dependencies installed.')

---
## Cell 3: Configure Output Paths
Symlink `outputs/` → Drive so checkpoints/logs persist across sessions.

In [ ]:
import os, sys

REPO_DIR     = '/content/svg-scaling-laws'
DRIVE_OUTPUTS = '/content/drive/MyDrive/svg-scaling-laws/outputs'
LOCAL_OUTPUTS = os.path.join(REPO_DIR, 'outputs')

os.makedirs(DRIVE_OUTPUTS, exist_ok=True)

if os.path.islink(LOCAL_OUTPUTS):
    os.unlink(LOCAL_OUTPUTS)
elif os.path.exists(LOCAL_OUTPUTS):
    import shutil
    shutil.rmtree(LOCAL_OUTPUTS)

os.symlink(DRIVE_OUTPUTS, LOCAL_OUTPUTS)
print(f'Symlink: {LOCAL_OUTPUTS} -> {DRIVE_OUTPUTS}')

# Ensure subdirs exist
for d in ['logs', 'checkpoints', 'plots', 'data/binary']:
    os.makedirs(os.path.join(DRIVE_OUTPUTS, d), exist_ok=True)

# Add repo to Python path
sys.path.insert(0, REPO_DIR)
print('Path configured.')

---
## Cell 4: Verify Binary Data & Print Model Parameter Counts
Confirm Phase 1 outputs are on Drive, then print exact param counts for all 5 models.

In [ ]:
import json, os
import numpy as np

# ---- Verify binary data ----
binary_dir = 'outputs/data/binary'
print('Binary files:')
for split in ['train', 'val', 'test']:
    p = os.path.join(binary_dir, f'{split}.bin')
    if os.path.exists(p):
        arr = np.memmap(p, dtype=np.uint16, mode='r')
        print(f'  {split}.bin: {len(arr):,} tokens  ({os.path.getsize(p)/1e6:.1f} MB)')
    else:
        print(f'  {split}.bin: MISSING — run Phase 1 first!')

split_info_path = os.path.join(binary_dir, 'split_info.json')
if os.path.exists(split_info_path):
    with open(split_info_path) as f:
        si = json.load(f)
    print(f'\nSplit info: {json.dumps(si, indent=2)}')

# ---- Model parameter counts ----
from src.model import MODEL_CONFIGS, TransformerLM, print_model_summary
print('\n' + '='*60)
print('MODEL PARAMETER COUNTS')
print('='*60)
print_model_summary()

---
## Cell 5: LR Sweep — Tiny Model
Trains Tiny at 7 learning rates (2000 steps each). Takes ~15-20 min on A100.

After this cell completes, read the best LR from the output and update Cell 6.

In [ ]:
!python scripts/06_lr_sweep.py --training_config configs/training_config.yaml

---
## Cell 6: Display LR Sweep Results & Set Best LR

In [ ]:
import json
from IPython.display import Image, display

with open('outputs/logs/lr_sweep_sp.json') as f:
    sweep = json.load(f)

print(f"Best LR: {sweep['best_lr']:.1e}  →  val_loss={sweep['best_val_loss']:.4f}")
print(f"\nAll runs:")
print(f"{'LR':>12} {'Val loss':>12} {'Diverged':>10}")
print("-" * 38)
for r in sorted(sweep['runs'], key=lambda x: x['lr']):
    marker = '*' if r['lr'] == sweep['best_lr'] else ' '
    print(f"{r['lr']:>12.1e} {r['val_loss']:>12.4f} {str(r['diverged']):>10} {marker}")

display(Image('outputs/plots/lr_sweep_sp.png'))

print(f"\n>>> Set learning_rate: {sweep['best_lr']:.1e} in configs/training_config.yaml before training")

In [ ]:
# Auto-update training_config.yaml with the best LR found above
import yaml, json

with open('outputs/logs/lr_sweep_sp.json') as f:
    sweep = json.load(f)

best_lr = sweep['best_lr']

with open('configs/training_config.yaml') as f:
    tcfg = yaml.safe_load(f)

tcfg['learning_rate'] = best_lr

with open('configs/training_config.yaml', 'w') as f:
    yaml.dump(tcfg, f, default_flow_style=False, allow_unicode=True)

print(f'configs/training_config.yaml updated: learning_rate = {best_lr:.1e}')

---
## Cell 7: Train — Tiny
~10 min on A100. Resumes from last checkpoint if Colab disconnected.

In [ ]:
!python scripts/05_train_model.py --model_name tiny --resume

In [ ]:
import json
with open('outputs/logs/result_tiny.json') as f:
    r = json.load(f)
print(f"Tiny complete.  Val loss: {r['best_val_loss']:.4f}.  "
      f"Time: {r['wall_time_min']:.1f} min.  Params: {r['n_params']:,}")

---
## Cell 8: Train — Small
~15 min on A100.

In [ ]:
!python scripts/05_train_model.py --model_name small --resume

In [ ]:
import json
with open('outputs/logs/result_small.json') as f:
    r = json.load(f)
print(f"Small complete.  Val loss: {r['best_val_loss']:.4f}.  "
      f"Time: {r['wall_time_min']:.1f} min.  Params: {r['n_params']:,}")

---
## Cell 9: Train — Medium
~30 min on A100.

In [ ]:
!python scripts/05_train_model.py --model_name medium --resume

In [ ]:
import json
with open('outputs/logs/result_medium.json') as f:
    r = json.load(f)
print(f"Medium complete.  Val loss: {r['best_val_loss']:.4f}.  "
      f"Time: {r['wall_time_min']:.1f} min.  Params: {r['n_params']:,}")

---
## Cell 10: Train — Large
~60 min on A100.

In [ ]:
!python scripts/05_train_model.py --model_name large --resume

In [ ]:
import json
with open('outputs/logs/result_large.json') as f:
    r = json.load(f)
print(f"Large complete.  Val loss: {r['best_val_loss']:.4f}.  "
      f"Time: {r['wall_time_min']:.1f} min.  Params: {r['n_params']:,}")

---
## Cell 11: Train — XL
~2-3 hours on A100. Save checkpoint to Drive frequently (every 500 steps).

In [ ]:
!python scripts/05_train_model.py --model_name xl --resume

In [ ]:
import json
with open('outputs/logs/result_xl.json') as f:
    r = json.load(f)
print(f"XL complete.  Val loss: {r['best_val_loss']:.4f}.  "
      f"Time: {r['wall_time_min']:.1f} min.  Params: {r['n_params']:,}")

---
## Cell 12: Summary — All Models Trained

In [ ]:
import json, os

model_names = ['tiny', 'small', 'medium', 'large', 'xl']
print(f"{'Model':<10} {'Params':>12} {'Best val L':>12} {'Time (min)':>12}")
print("-" * 52)

all_done = True
for name in model_names:
    p = f'outputs/logs/result_{name}.json'
    if os.path.exists(p):
        with open(p) as f:
            r = json.load(f)
        print(f"{name:<10} {r['n_params']:>12,} {r['best_val_loss']:>12.4f} "
              f"{r.get('wall_time_min', '?'):>12}")
    else:
        print(f"{name:<10} {'NOT TRAINED':>38}")
        all_done = False

if all_done:
    print("\nAll 5 models trained. Ready to fit scaling law.")
else:
    print("\nSome models still need training. Run the corresponding cells above.")

---
## Cell 13: Fit Scaling Law & Generate All Plots

In [ ]:
!python scripts/11_plot_results.py

---
## Cell 14: Display All Plots

In [ ]:
from IPython.display import Image, display
import os

plots = [
    ('Scaling Law (SP)', 'outputs/plots/scaling_law_sp.png'),
    ('Training Curves',  'outputs/plots/training_curves.png'),
    ('LR Sweep',         'outputs/plots/lr_sweep_sp.png'),
]

for title, path in plots:
    if os.path.exists(path):
        print(f'\n--- {title} ---')
        display(Image(path))
    else:
        print(f'Missing: {path}')

---
## Cell 15: Scaling Law Summary

In [ ]:
import json, os

fit_path = 'outputs/logs/scaling_fit_sp.json'
if os.path.exists(fit_path):
    with open(fit_path) as f:
        fit = json.load(f)

    print('='*60)
    print('SCALING LAW FIT (Standard Parameterization)')
    print('='*60)
    print(f"  L = {fit['a']:.4f} * N^(-{fit['alpha']:.4f}) + {fit['c']:.4f}")
    print(f"  alpha   = {fit['alpha']:.4f}  (Kaplan NL: 0.076)")
    print(f"  R²      = {fit['r_squared']:.4f}")

    ex = fit.get('extrapolation_10x', {})
    if ex:
        print(f"\nExtrapolation (10× XL ≈ {ex['N']/1e6:.0f}M params):")
        print(f"  Predicted L = {ex['L_pred']:.4f}")
        print(f"  95% CI:       [{ex['L_lower']:.4f}, {ex['L_upper']:.4f}]")
else:
    print('scaling_fit_sp.json not found — run Cell 13 first.')

---
## Done!

Phase 2 outputs saved to Drive:
| File | Purpose |
|------|---------|
| `outputs/checkpoints/<model>/best.pt` | Best checkpoint per model |
| `outputs/logs/training_<model>.csv` | Step-level training logs |
| `outputs/logs/result_<model>.json` | Final result per model |
| `outputs/logs/lr_sweep_sp.json` | LR sweep results |
| `outputs/logs/scaling_fit_sp.json` | Fitted power law parameters |
| `outputs/plots/scaling_law_sp.png` | Main scaling plot |
| `outputs/plots/training_curves.png` | Training curves all models |
| `outputs/plots/lr_sweep_sp.png` | LR sweep U-curve |

**Next:** Phase 3 — µP parameterization (`03_mup_study.ipynb`)